# iSCORS — axes summary (clean)

Single load at the top, then three tabs:
1. **Paper replication** — gamma, CV², slope-3 condensation, y-intercept.
2. **Single-cell most-reproducible axis** — reliability-max (cross-half GEVD), X⊥Y residual.
3. **Multi-axis decomposition** — G(τ) GEVD axis 1/2, de-nuisanced ICA comp 1/2.

Notes:
- **ICA order & sign are arbitrary** (FastICA does not sort or fix sign). We fix a convention:
  sparse (high |kurtosis|) component = comp 1, heavy tail positive — so runs are reproducible.
- **De-nuisance basis = poly3 + radial vignetting** (kept identical wherever ICA is used, so the
  input subspace — hence the ICA result — does not drift between cells).
- The temporal flat-field cancels in C(τ)/C(0), so g_norm is insensitive to it.


In [ ]:
# imports, config, loaders
import os, gc, sys, zipfile, numpy as np, matplotlib.pyplot as plt, tifffile
from scipy.ndimage import gaussian_filter, zoom
from scipy.linalg import eigh as geigh
from scipy.stats import pearsonr, kurtosis
from sklearn.decomposition import FastICA

REPO = '/content/iscors-net'
if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path: sys.path.insert(0, REPO)
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density, compute_g_norm_torch

ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or .tif
VIDEO_FNAME = 'COBRI_rarw_video.tif'
EXTRACT_DIR = '/content/real_data'
MASK_PATH   = 'data/condensation_mask.tif'
N_FRAMES = 5000; BIN = 2; SLOPE = 3.0; GAMMA_SCALE = 2.0
RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)

def load_video(n=N_FRAMES, b=BIN):
    p = ZIP_PATH
    if str(p).lower().endswith('.zip'):
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        find = lambda: next((os.path.join(dp, VIDEO_FNAME) for dp, _, fs in os.walk(EXTRACT_DIR) if VIDEO_FNAME in fs), None)
        vp = find()
        if not vp:
            with zipfile.ZipFile(p) as z: z.extractall(EXTRACT_DIR)
            vp = find()
    else:
        vp = p
    H0, W0 = tifffile.imread(vp, key=0).shape
    with tifffile.TiffFile(vp) as tf:
        try: total = int(tf.series[0].shape[0])
        except Exception: total = len(tf.pages)
    n = min(n, total); Hb, Wb = H0 // b, W0 // b; Hc, Wc = Hb * b, Wb * b
    v = np.empty((n, Hb, Wb), np.float32)
    for s in range(0, n, 200):
        e = min(s + 200, n); ch = tifffile.imread(vp, key=range(s, e)).astype(np.float32)
        v[s:e] = ch[:, :Hc, :Wc].reshape(e - s, Hb, b, Wb, b).mean((2, 4))
    return v

def preprocess_inplace(v):
    v /= (v.mean(0) + 1e-10)                                  # temporal flat-field (cancels in g_norm)
    for t in range(len(v)): v[t] /= (gaussian_filter(v[t], 4) + 1e-10)   # per-frame BG
    return v

def load_mask(shape):
    mk = np.asarray(tifffile.imread(MASK_PATH)).squeeze()
    if mk.ndim == 3: mk = mk[..., 0]
    if mk.shape != shape: mk = zoom(mk, (shape[0] / mk.shape[0], shape[1] / mk.shape[1]), order=0)
    bd = np.zeros(shape, bool); bd[0, :] = bd[-1, :] = bd[:, 0] = bd[:, -1] = True
    return mk == min(np.unique(mk), key=lambda z: (mk[bd] == z).mean())   # interior level = nucleus


In [ ]:
# === single load: compute every per-pixel map once, then free the video ===
video = preprocess_inplace(load_video()); h = len(video) // 2
def _maps(seg):
    f = gpu_fit_maps(seg, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                     gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=400, verbose=False)
    d, _ = compute_density(seg, min_cv=0.005)
    g, _c, _z = compute_g_norm_torch(seg, RECON_TAUS, norm='nor1', min_cv=0.005)
    return f['gamma'].astype(np.float32), d.astype(np.float32), g.cpu().numpy().astype(np.float32)
gamma, dens, G = _maps(video)
(g1g, g1d, G1), (g2g, g2d, G2) = _maps(video[:h]), _maps(video[h:])
nuc = load_mask(dens.shape) & np.isfinite(gamma)
del video; gc.collect()
_lg = lambda a: np.log10(np.clip(a, 1e-12, None))
X,  Y  = _lg(1 / np.clip(gamma, 1e-6, None)), _lg(dens)
X1, Y1 = _lg(1 / np.clip(g1g, 1e-6, None)), _lg(g1d)
X2, Y2 = _lg(1 / np.clip(g2g, 1e-6, None)), _lg(g2d)
print('setup done, video freed. nucleus px:', int(nuc.sum()))


In [ ]:
# helpers
m = nuc & np.isfinite(X) & np.isfinite(Y)
def _full(vec_on_m):
    z = np.full(nuc.shape, np.nan); z[m] = vec_on_m; return z
def _gevd(a1, a2):
    Cg = (a1.T @ a2) / a1.shape[0]; Cg = (Cg + Cg.T) / 2
    v, w = geigh(Cg, np.cov(np.vstack([a1, a2]).T)); o = np.argsort(v)[::-1]
    return v[o], w[:, o]
def show(items, title):
    n = len(items); fig, ax = plt.subplots(1, n, figsize=(4.2 * n, 4.2))
    ax = np.atleast_1d(ax)
    for a, (t, d, cm) in zip(ax, items):
        im = a.imshow(d, cmap=cm, vmin=np.nanpercentile(d, 2), vmax=np.nanpercentile(d, 98))
        a.set_title(t, fontsize=10); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(title); plt.tight_layout(); plt.show()


In [ ]:
# === Tab 1 — paper replication ===
b3 = float((Y[m] - SLOPE * X[m]).mean())
cond = _full(((X + SLOPE * Y - SLOPE * b3) / (1 + SLOPE ** 2))[m])
show([('gamma (log)',               _full(_lg(np.clip(gamma, 1e-6, None))[m]), 'inferno'),
      ('CV² (log)',                 _full(Y[m]),                               'inferno'),
      ('slope-3 condensation',      cond,                                      'inferno'),
      ('y-intercept density (Y-3X)',_full((Y - SLOPE * X)[m]),                 'inferno')],
     'Tab 1 - paper replication')


In [ ]:
# === Tab 2 — single-cell most-reproducible axis ===
mC = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
mx, my, sx, sy = X[mC].mean(), Y[mC].mean(), X[mC].std(), Y[mC].std()
f1 = np.vstack([(X1[mC] - mx) / sx, (Y1[mC] - my) / sy]); f2 = np.vstack([(X2[mC] - mx) / sx, (Y2[mC] - my) / sy])
_, Wr = _gevd(f1, f2); vrel = Wr[:, 0]
relmax = _full((vrel[0] * (X - mx) / sx + vrel[1] * (Y - my) / sy)[m])
a, b = np.polyfit(Y[m], X[m], 1); xperp = _full((X - (a * Y + b))[m])
show([('reliability-max axis', relmax, 'inferno'),
      ('X-perp-Y residual',    xperp,  'coolwarm')],
     'Tab 2 - single-cell reproducible axis')


In [ ]:
# === Tab 3 — multi-axis decomposition (G(tau) GEVD + de-nuisanced ICA) ===
feat = lambda g, yl: np.concatenate([g, yl[..., None]], -1)
F1, F2, Ff = feat(G1, Y1)[m], feat(G2, Y2)[m], feat(G, Y)[m]
mu = (F1.mean(0) + F2.mean(0)) / 2; sd = (F1.std(0) + F2.std(0)) / 2 + 1e-9
_, W = _gevd((F1 - mu) / sd, (F2 - mu) / sd); pj = ((Ff - mu) / sd) @ W

yy, xx = np.mgrid[0:nuc.shape[0], 0:nuc.shape[1]]
xa = (xx[m] - xx[m].mean()) / xx[m].std(); ya = (yy[m] - yy[m].mean()) / yy[m].std()
D = np.vstack([np.ones_like(xa), xa, ya, xa**2, ya**2, xa*ya, xa**3, ya**3, xa**2*ya, xa*ya**2,
               np.sqrt(xa**2 + ya**2), xa**2 + ya**2]).T            # poly3 + radial vignetting
Pinv = np.linalg.pinv(D); dn = lambda Fm: Fm - D @ (Pinv @ Fm)
_, W2 = _gevd((dn(F1) - mu) / sd, (dn(F2) - mu) / sd)
S = FastICA(2, random_state=0, whiten='unit-variance', max_iter=1000).fit_transform(((dn(Ff) - mu) / sd) @ W2[:, :2])
S = S[:, np.argsort([-abs(kurtosis(S[:, 0])), -abs(kurtosis(S[:, 1]))])]   # sparse component first
for j in range(2):
    if abs(S[:, j].min()) > abs(S[:, j].max()): S[:, j] = -S[:, j]         # heavy tail positive
show([('G(tau) GEVD axis 1', _full(pj[:, 0]), 'inferno'), ('G(tau) GEVD axis 2', _full(pj[:, 1]), 'inferno'),
      ('ICA comp 1 (condensate)', _full(S[:, 0]), 'inferno'), ('ICA comp 2', _full(S[:, 1]), 'inferno')],
     'Tab 3 - multi-axis decomposition')
